In [1]:
# ===============================
# CONTROL PANEL
# ===============================

SEQ_LEN = 6

FEATURE_COLS = [
    "LAT",
    "LON",
    "SPEED",
    "dt",
    "COG_cos",
    "COG_sin",
    "HEADING_cos",
    "HEADING_sin"
]

TARGET_COLS = ["dlat", "dlon"]

BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 0.001

TRAIN_SPLIT_RANDOM_STATE = 42

In [2]:
# ===============================
# IMPORTS
# ===============================

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split

In [4]:
# ===============================
# LOAD DATA
# ===============================

train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

In [5]:
# ===============================
# FEATURE ENGINEERING
# ===============================

def convert_time(df):
    df["TIME"] = pd.to_datetime(df["TIME"])
    return df


def compute_trig(df):

    df["COG_rad"] = np.deg2rad(df["COG"])
    df["COG_cos"] = np.cos(df["COG_rad"])
    df["COG_sin"] = np.sin(df["COG_rad"])

    df["HEADING_rad"] = np.deg2rad(df["HEADING"])
    df["HEADING_cos"] = np.cos(df["HEADING_rad"])
    df["HEADING_sin"] = np.sin(df["HEADING_rad"])

    return df


def compute_deltas(df):

    df["dlat"] = df.groupby("voyage_id")["LAT"].shift(-1) - df["LAT"]
    df["dlon"] = df.groupby("voyage_id")["LON"].shift(-1) - df["LON"]

    df = df.dropna(subset=["dlat","dlon"])

    return df

In [6]:
# ===============================
# APPLY FEATURE ENGINEERING
# ===============================

train_df = convert_time(train_df)
test_df = convert_time(test_df)

train_df = compute_trig(train_df)
test_df = compute_trig(test_df)

train_df = compute_deltas(train_df)
test_df = compute_deltas(test_df)

In [7]:
# ===============================
# SEQUENCE GENERATION
# ===============================

def create_sequences(df, seq_len):

    X = []
    Y = []

    for _, trip in df.groupby("voyage_id"):

        trip = trip.sort_values("TIME")

        if len(trip) < seq_len + 1:
            continue

        features = trip[FEATURE_COLS].values
        targets = trip[TARGET_COLS].values

        for i in range(len(trip) - seq_len):

            X.append(features[i:i+seq_len])
            Y.append(targets[i+seq_len])

    return np.array(X), np.array(Y)

In [8]:
# ===============================
# BUILD DATASETS
# ===============================

X_train, Y_train = create_sequences(train_df, SEQ_LEN)
X_test, Y_test = create_sequences(test_df, SEQ_LEN)

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)

X_train shape: (13083, 6, 8)
Y_train shape: (13083, 2)


In [9]:
# ===============================
# TORCH DATASETS
# ===============================

X_train = torch.tensor(X_train, dtype=torch.float32)
Y_train = torch.tensor(Y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
Y_test = torch.tensor(Y_test, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train, Y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, Y_test),
    batch_size=BATCH_SIZE
)

In [21]:
# ===============================
# TEMPORAL BLOCK
# ===============================

class TemporalBlock(nn.Module):

    def __init__(self, in_channels, out_channels, kernel_size, dilation):

        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.relu2 = nn.ReLU()

        self.downsample = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else None
        )

    def forward(self, x):

        seq_len = x.size(2)

        out = self.conv1(x)
        out = out[:, :, :seq_len]      # crop
        out = self.relu1(out)

        out = self.conv2(out)
        out = out[:, :, :seq_len]      # crop again
        out = self.relu2(out)

        res = x if self.downsample is None else self.downsample(x)

        return out + res

In [22]:
# ===============================
# MODEL
# ===============================

class TCN(nn.Module):

    def __init__(self, input_dim, output_dim):

        super().__init__()

        self.tcn = nn.Sequential(
            TemporalBlock(input_dim, 32, kernel_size=3, dilation=1),
            TemporalBlock(32, 64, kernel_size=3, dilation=2),
            TemporalBlock(64, 64, kernel_size=3, dilation=4)
        )

        self.fc = nn.Linear(64, output_dim)


    def forward(self, x):

        # convert (batch, seq, features) → (batch, features, seq)
        x = x.transpose(1, 2)

        # run through stacked temporal blocks
        y = self.tcn(x)

        # take final timestep
        y = y[:, :, -1]

        # map to prediction
        return self.fc(y)

In [ ]:
# ===============================
# TRAIN MODEL
# ===============================

model = TCN(input_dim=len(FEATURE_COLS), output_dim=2)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

criterion = nn.MSELoss()


for epoch in range(EPOCHS):

    model.train()

    for X_batch, Y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(pred, Y_batch)

        loss.backward()

        optimizer.step()

    print("Epoch:", epoch, "Loss:", loss.item())

/home/jacobhardy/miniconda3/lib/python3.13/site-packages/torch/autograd/graph.py:841: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch: 0 Loss: 118633.21875
Epoch: 1 Loss: 5.293631553649902
Epoch: 2 Loss: 1.0964751243591309
Epoch: 3 Loss: 13.67125129699707
Epoch: 4 Loss: 3736.6845703125
Epoch: 5 Loss: 3352.93896484375
Epoch: 6 Loss: 238.64772033691406
Epoch: 7 Loss: 333.80133056640625
Epoch: 8 Loss: 1.150612473487854
Epoch: 9 Loss: 26.545265197753906
Epoch: 10 Loss: 2.5291199684143066
Epoch: 11 Loss: 3.28952956199646
Epoch: 12 Loss: 0.1452028751373291
Epoch: 13 Loss: 0.8431475758552551
Epoch: 14 Loss: 1.7581071853637695
Epoch: 15 Loss: 2.634770631790161


In [ ]:
# ===============================
# EVALUATION METRICS
# ===============================

model.eval()

with torch.no_grad():
    preds = model(X_test)

preds = preds.numpy()
truth = Y_test.numpy()


# last known vessel position in the sequence
last_lat = X_test[:, -1, 0].numpy()
last_lon = X_test[:, -1, 1].numpy()


# predicted position
pred_lat = last_lat + preds[:, 0]
pred_lon = last_lon + preds[:, 1]


# true position
true_lat = last_lat + truth[:, 0]
true_lon = last_lon + truth[:, 1]


# -------------------------------
# haversine distance (meters)
# -------------------------------

R = 6371000

lat1 = np.radians(true_lat)
lat2 = np.radians(pred_lat)

dlat = lat2 - lat1
dlon = np.radians(pred_lon - true_lon)

a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
c = 2*np.arctan2(np.sqrt(a), np.sqrt(1-a))

errors = R * c


rmse = np.sqrt(np.mean(errors**2))
mae = np.mean(np.abs(errors))


print("RMSE (meters):", rmse)
print("MAE (meters):", mae)